<a href="https://colab.research.google.com/github/DivyaMeenaSundaram/Prompt-Engineering/blob/main/Jinja2_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A prompt and an LLM are two different things.

A prompt is an instruction/input.

An LLM is the model that generates the answer.

In [1]:
prompt = """
You are a Walmart customer support assistant.

Answer the customer's question accurately
and concisely.

Customer question:
Can I return this product after 45 days?
"""

In [2]:
print(prompt)


You are a Walmart customer support assistant.

Answer the customer's question accurately
and concisely.

Customer question:
Can I return this product after 45 days?



Trial LLM call

In [3]:
# Install the LangChain integration for Google's Gemini models.
!pip install -q -U langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 13.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [4]:
# Import os so that we can store the API key as an environment variable.
import os

# Import getpass so that the API key is not displayed while typing it.
from getpass import getpass

# Ask the user to enter the Gemini API key securely.
os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [5]:
# Import the LangChain class that allows us to communicate with Gemini.
from langchain_google_genai import ChatGoogleGenerativeAI

# Create a Gemini LLM object.
# This object represents the AI model that will generate our answers.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash"
)

In [8]:
# LLM is the connection/interface to the AI model. It represents the model we are going to send prompts to.
llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-google-genai': '4.4.0'}}, profile={}, google_api_key=SecretStr('**********'), model='gemini-3.8-flash', temperature=None, client=<google.genai.client.Client object at 0x7e29e386b0e0>, default_metadata=(), model_kwargs={})

In [10]:
# Send a question directly to the Gemini model.
response = llm.invoke(
    "Can I return an opened laptop after 60 days?"
)

# Print only the text generated by the model.
print(response.content)

[{'type': 'text', 'text': "In most cases, **no**, you cannot return an opened laptop after 60 days. For electronics, most major retailers limit returns to **14 to 30 days**. \n\nHowever, whether you can return it depends entirely on **where you bought it**. Here is a breakdown of retailer policies and potential alternatives:\n\n---\n\n### Exceptions Where You **Can** Return It at 60 Days:\n* **Costco:** Offers a **90-day return window** for electronics, including laptops (receipt and all original accessories required).\n* **Best Buy:** Standard return policy is only 14 days, but **My Best Buy Plus™ and Total™ members** get a **60-day return window**.\n* **Holiday Return Windows:** If you bought the laptop in October or November, many retailers (like Amazon, Best Buy, Walmart, and Target) extend return deadlines until mid-to-late January.\n\n---\n\n### Stores Where You **Cannot** Return It at 60 Days:\n* **Amazon:** 30 days\n* **Apple Store:** 14 days\n* **Target:** 15 days\n* **Walmart

## Why Do We Need Prompt Templates?

So far, the prompt has been written manually as a complete block of text. This works well when there is only one customer question. However, in a real application, the same type of prompt may need to be used thousands of times with different customer questions.

For example, consider the following prompt:

```text
You are a Walmart customer support assistant.

Answer the customer's question accurately
and concisely.

Customer question:
Can I return an opened laptop after 60 days?
```

Tomorrow, a different customer may ask:

```text
My order says delivered but I haven't received it.
```

The **instructions do not change**. The only thing that changes is the customer's question.

Manually rewriting the entire prompt every time would be unnecessary and error-prone. Instead, the prompt can be divided into two parts.

### 1. Fixed Part

The fixed part contains the instructions that remain the same for every customer:

```text
You are a Walmart customer support assistant.

Answer the customer's question accurately
and concisely.

Customer question:
```

This part defines **how the model should behave** and provides the context in which the user's question should be interpreted.

### 2. Variable Part

The variable part contains the information that changes from one request to another:

```text
Can I return an opened laptop after 60 days?
```

For another customer, the variable could be:

```text
My order says delivered but I haven't received it.
```

Therefore, instead of creating a completely new prompt each time, the application can keep the fixed instructions and insert a new customer question whenever required.

The basic structure is:

```text
FIXED PROMPT
     +
VARIABLE INPUT
     ↓
FINAL PROMPT
```

For example:

```text
Fixed Prompt:
"You are a Walmart customer support assistant.
 Answer the customer's question accurately and concisely.
 Customer question:"

        +

Variable:
"Can I return an opened laptop after 60 days?"

        ↓

Final Prompt:
"You are a Walmart customer support assistant.
 Answer the customer's question accurately and concisely.
 Customer question:
 Can I return an opened laptop after 60 days?"
```

This is the fundamental idea behind **prompt templating**.

A **prompt template** is simply a reusable prompt structure in which the instructions remain fixed while specific values are inserted dynamically.

In code, the variable portion is given a name, such as `{customer_question}`:

```text
You are a Walmart customer support assistant.

Answer the customer's question accurately
and concisely.

Customer question:
{customer_question}
```

Now the same template can be reused with different inputs:

```text
{customer_question}
        ↓
"Can I return an opened laptop after 60 days?"
```

or

```text
{customer_question}
        ↓
"My order says delivered but I haven't received it."
```

The template therefore provides **consistency, reusability, and dynamic input handling**.

This is exactly the problem that tools such as LangChain's `PromptTemplate` are designed to solve.



#                                                                     Jinja2



In [11]:
# Install Jinja2, a Python templating engine.
!pip install -q jinja2

In [12]:
# Import the Template class from Jinja2.
from jinja2 import Template

# Create a reusable prompt template.
# {{ customer_question }} is a placeholder for dynamic information.
template = Template("""
You are a Walmart customer support assistant.

Answer the customer's question accurately and concisely.

Customer question:
{{ customer_question }}
""")

In [13]:
# Provide the actual customer question that should replace
# {{ customer_question }} in the template.
customer_question = "Can I return an opened laptop after 60 days?"

# Render the template.
# Jinja2 replaces {{ customer_question }} with the actual question.
prompt = template.render(
    customer_question=customer_question
)

# Display the resulting prompt.
print(prompt)


You are a Walmart customer support assistant.

Answer the customer's question accurately and concisely.

Customer question:
Can I return an opened laptop after 60 days?


# Jinja2 + LLM

In [14]:
# Import Jinja2's Template class.
from jinja2 import Template

# Create a reusable prompt template.
template = Template("""
You are a Walmart customer support assistant.

Answer the customer's question accurately and concisely.

Customer question:
{{ customer_question }}
""")

# Ask the user to enter a customer question.
customer_question = input("Customer: ")

# Insert the customer's question into the Jinja2 template.
prompt = template.render(
    customer_question=customer_question
)

# Send the completed prompt to the Gemini LLM.
response = llm.invoke(prompt)

# Display the AI-generated answer.
print("\nWalmart Assistant:", response.content)

Customer: can i return a used product ?

Walmart Assistant: [{'type': 'text', 'text': 'Yes, in most cases, you can return an opened or gently used item. \n\nHere are the general guidelines:\n* **Standard Return Window:** Most items can be returned within **90 days** of purchase (**30 days** for most electronics).\n* **Requirements:** Please bring the item with its original packaging, accessories, and your receipt or order number.\n* **Exceptions:** Certain opened items (such as video games, computer software, airbeds, or prescription items) cannot be returned for a refund and may only be eligible for an exchange.\n\nYou can return items at the Customer Service desk in any Walmart store or start a return online through your Walmart account.', 'extras': {'signature': 'EtEOCs4OARFNMg9hEpjO53o7uiKgowDY8DP47myX1e9YF1Eo4Vpm204lJUreNpYha2LWOd1p7mQCsgdFY1/Nf+FT+hiTnhDXYS0VPHxOHBPYpSHnorxgZ+Gq3T0foMiC3B2itVUBb9lldidPSejmbyBeyjKktKODnc/paFZCtDRoSHt7iAjY6JaE9ueWWmiHMEdY4qTy5NX+7hYuACK5gpMpRRKnR3Q

## Adding Context to a Prompt Template

The previous example used a prompt template with one variable:

```text
{{ customer_question }}
```

However, a customer-support assistant should not always rely only on its general knowledge to answer a question.

In a real customer-support system, the assistant may be provided with **relevant information or context** that should be used when generating the answer.

For example, suppose the customer asks:

```text
Can I return an opened laptop after 60 days?
```

Instead of asking the language model to answer this question entirely from its general knowledge, the application can provide relevant return-policy information:

```text
Context:
Returns are subject to applicable eligibility conditions.
Some product categories have different requirements.
If eligibility cannot be determined, additional information
is required.
```

The model can then use this context when answering the customer's question.

### From One Variable to Two Variables

The prompt template now contains **two pieces of information that can change**:

1. `context` — the relevant information provided to the model.
2. `customer_question` — the question asked by the customer.

The template combines these two inputs with the fixed instructions:

```text
FIXED INSTRUCTIONS
        +
CONTEXT
        +
CUSTOMER QUESTION
        ↓
FINAL PROMPT
```

In [15]:
# Import Jinja2's Template class.
from jinja2 import Template

# Create a reusable prompt template with two dynamic variables:
# 1. context = information available to the AI
# 2. customer_question = the customer's actual question
template = Template("""
You are a Walmart customer support assistant.

Use only the supplied context to answer the question.

If the context does not contain enough information,
say that additional information is required.

Context:
{{ context }}

Customer question:
{{ customer_question }}

Answer:
""")

# Store the information available to the AI.
context = """
Returns are subject to applicable eligibility conditions.

Some product categories have different requirements.

If eligibility cannot be determined from the available
information, additional information is required.
"""

# Ask the user for a customer question in real time.
customer_question = input("Customer: ")

# Insert both the context and customer question into the template.
prompt = template.render(
    context=context,
    customer_question=customer_question
)

# Send the completed prompt to Gemini.
response = llm.invoke(prompt)

# Print the AI's answer.
print("\nWalmart Assistant:", response.content)

Customer: can i return a used item after 30 days 

Walmart Assistant: [{'type': 'text', 'text': 'Additional information is required.', 'extras': {'signature': 'EtgMCtUMARFNMg/QZ0s3Gg4gXqplhyl7RkrEtpo/5Bo8dL6dV4B1fjowUvAAOCk7wS4xuCH+4L9d2A7TeT+8baEg8anfecYn7JCQSyjfuWBFIVbK6HPSnjF7naO7a7HGB3TAmI0zVq73e/1/yRqgqZzXNULYsZQC03gRDY4FjLjjKR9YfnwaqyES4Ph3wIgHhM+mhy66Mlpu8GOtPOqposnL2g31KkbKhPSBdV5X/4F0BB0MAEpKKKWi/VnvT3ztasnm8qY5yhhaftXxRes+l1Q71PasoFQHXNGSFp2IneDSuhap9UpUYdaYjz8PN6/2ci4Le+wXwSDvE5uV1EymlKJBHCdppfnCbe32HYFpolw/0AXoUIdG8RbvHNG+7RhJe2bYgmkJLDxHqhS0hnsmLO66eLHCfJK1yVUmaLepBFLQXhU32PVlKa491ZrmtPrnIM1v8DEv4hfAYQSXUJpXeyzl8OKx3n3LrsDRAkpcL/lfSe+5zAb6Adc7nWyTJI/hRLeTWcxUkEsVhzdSWfquYssA1WauhPnGyyQ1/rCJNSr/VDW/ws1sxX6WwuY64jZ/kbYQQlKmqFk9Uzg7kuEatRCF7WvxdYjOZCrpCTOa4sdeKi6azlh3hpqjRNhrR8hdax4SnZW6wtDZrqwYL+9/PxF26XMFi6/k6TQLEXhCwb4jSZzsvzVE/9xKvPwtgfm9euzbgU+xTnAFgi+LuA7T/oZ3z8su6uti2AvUu3Ms2peeZWOhmxKL2eIlZRzQ0cm7vOD7fsTv36adjlNmRZfpQ88G/2F6z80UhptVz4UNSb3ZgRR0+I2otmRO3wsVnyGGJXK5tKNr

# LangChain PromptTemplate

In [16]:
# Import PromptTemplate from LangChain.
# PromptTemplate helps us create reusable text prompts
# with variables that can be filled at runtime.
from langchain_core.prompts import PromptTemplate

In [17]:
# Create a reusable LangChain prompt template.
prompt_template = PromptTemplate(

    # Tell LangChain which variables will be supplied later.
    input_variables=["question", "context"],

    # Define the actual prompt structure.
    # {question} and {context} are placeholders.
    template="""
You are a Walmart customer support assistant.

Use only the supplied context to answer the question.

If the context does not contain enough information,
say that additional information is required.

Context:
{context}

Customer question:
{question}

Answer:
"""
)

In [18]:
# Ask the customer to enter their question.
customer_question = input("Customer: ")

# Provide the information that the AI is allowed to use.
context = """
Returns are subject to applicable eligibility conditions.

Some product categories have different requirements.

If eligibility cannot be determined from the available
information, additional information is required.
"""

# Fill the placeholders in the LangChain template.
# {question} is replaced by customer_question.
# {context} is replaced by context.
prompt = prompt_template.format(
    question=customer_question,
    context=context
)

# Display the final prompt that will be sent to the LLM.
print("\nGenerated Prompt:\n")
print(prompt)

Customer: can i return a used item ?

Generated Prompt:


You are a Walmart customer support assistant.

Use only the supplied context to answer the question.

If the context does not contain enough information,
say that additional information is required.

Context:

Returns are subject to applicable eligibility conditions.

Some product categories have different requirements.

If eligibility cannot be determined from the available
information, additional information is required.


Customer question:
can i return a used item ?

Answer:



In [19]:
# Send the generated prompt to the Gemini LLM.
response = llm.invoke(prompt)

# Display the AI-generated answer.
print("\nWalmart Assistant:", response.content)


Walmart Assistant: [{'type': 'text', 'text': 'Additional information is required to determine if your item can be returned. Returns are subject to applicable eligibility conditions, and some product categories have different requirements.', 'extras': {'signature': 'EvsLCvgLARFNMg8c5H3a0aJq7LtPIFKwFDxjtCxzuSAh+PI/hyoeqOpOmFiyG2AM08J85Ll8NxzPa6GCIgkUYDza7eIMWpMsPS8/m3ix0g1ZnK76tMz0JHbMOHzHMVcO2U2OPbD8mG9IUbDgWx/xrKu4ZkK4tRWkaoiEts21508bVH9ojLUZnaJNF3Cit65L+XFqw0zNUvoAXfx974BYiDk4mmCYz6lC0JYPYYABUw3C7vrbBMJN9E3h0Qq+VKfif3taaKjtKPiSbLghHGZr+WBrYJ7sT6l86wQmSfIj4EVJKIHu9LJ/lQ4c2nvxQ/3ZhFlAWl+XgKG89Qidj+rC36OTMX1q6lz4NC06Mcfr9apNTtIGlaO4EQ0H+edWw2PvzcWXmwkNBBa2fSf/yEoz24CC95ytZOgeY1mBPJ9/uT9WAEbrvlTNVQoNSbX/vsNpX66AF77rcCBRAEd+eHMw0GXOmXK40gAQel+iPdMDF1OvzlO4M2dpl4g+n9WHPvwF4a+eoo76KxWcItz37lQHN6ZVUYVHg1XZWAbrIrdkUC/27HDEhPEXk5BCygiopLZpESl/vOHS0jZOl81fkywphcuqQVAR+33ApHIB5FKKPniCZ6RZCQgdDv7Ho2vf5yiBE/5CjRSVQam4l6Q5yVh6tYPpKO0TiyPLuQ5P9SivAQfPQ3RsxCcYCmY8JJk5wqawomxiwEoviKcKoBjyl1kDsNZGjwrN5

Option 2

In [20]:
# Connect the prompt template directly to the LLM.
# The | operator means:
# "Take the output from the prompt and pass it to the LLM."
chain = prompt_template | llm

In [21]:
# Ask the customer for a question.
customer_question = input("Customer: ")

# Send the variable values into the chain.
# LangChain automatically:
# 1. fills the prompt template
# 2. creates the final prompt
# 3. sends it to Gemini
response = chain.invoke({
    "question": customer_question,
    "context": context
})

# Print the final AI response.
print("\nWalmart Assistant:", response.content)

Customer: can i return an unopened item after 2 days with receipt ?

Walmart Assistant: [{'type': 'text', 'text': 'Additional information is required.', 'extras': {'signature': 'Et4MCtsMARFNMg8tPJw2CR5vpunZk40ogF20puPt4LbH9b75WeNkp13wnJ94c1TJBPglZMhqDK/KWi0ins+LtJkGsJvSC/OTtWUuyWe00K0MNJtTmXKFSyG2umnkt2lDpvKsiNgZ3cBvYaytDU/2sZZuP+qy5TMW55A2IzWiVEHjwNkJgOsRGZwzrYuYPkZO4asugNxzBsxVEW3+w6wab9JknINo+41VheKRg/wg8/InWFfZxY863csD+pL/Dh7z1YHVep+HlWZLshr4EjQz8yqE9RZLRV/3Cvz9TucESrNC4+ICMRZ95/ZtVfxSkcz2SY0V/sFEnbm+NleVHEvNCq5uDz0++UvmZThfJlySD4qM3fY6siot/iS8vyvCjV5KGeBD52coHCQ6zAAUCRkxQNEaRaTxiEnRt97rT188PZtqUD6ya9g9TgWJo95jR13D8C99gzAzGyCWRGX/62eWlCHCkWuWOh7kMQLrT7TBXTyDhG7R1tSeaEHLpRP78lJrpfmPHnwGZFdg8ucJvTeur1yYWe5gX+rMFR75GMVmaBKW9Yy4Xy2pV03uM+ByJkqm2AIRix7w9E180Iv+EZTezZTW+9U8KjCmzoy0VrHWMEgWHIE3l/CS7DMHcTjUhbo7834KZKIgzYat4xoHpp3gJcI626arlsvtrO0pFj/R8hy1jF8EuiU0nuITJv2KzyM6Ao7sCsiIw3qELjSi8qc84jzuqLQ0u1xJlYJL+9XN/v+87ByHcGD5VfBgU7AlgNTomJob7wuujoYFG4fpZk+APqRbVXU7/hPDPrImRnIfqb7NWs4iEkB6TF

#  `ChatPromptTemplate`

So far, `PromptTemplate` has allowed us to create reusable prompts by separating **fixed instructions** from **variable inputs**. This works well when the entire prompt can be treated as **one block of text**.

However, modern LLM applications are usually not structured as one large block of text. Instead, conversations are organized into **messages**, and each message has a specific **role**.



## Why Do Roles Matter?

Consider a typical interaction with a chat-based AI system:

```text
SYSTEM
You are a Walmart customer support assistant.
Answer customer questions accurately and concisely.

HUMAN
Can I return this product?

AI
Yes, provided the product meets the applicable return requirements.
```

There are three distinct roles here:

### System: The **system message** defines the behavior, instructions, constraints, and overall role of the AI assistant.

### Human: The **human message** represents the user's actual request.

### AI: The **AI message** represents a response generated by the assistant.


# `ChatPromptTemplate`

LangChain provides `ChatPromptTemplate` for creating these structured chat prompts. It allows us to define different message roles directly in the template.

For example:

```python
chat_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a Walmart customer support assistant. "
        "Answer customer questions accurately and concisely."
    ),
    (
        "human",
        "{customer_question}"
    )
])
```
Here, the prompt is no longer one giant text block. Instead, it consists of two messages: System message and human question.



## Adding Context to `ChatPromptTemplate`

The same idea of dynamic context can also be incorporated into a chat prompt.
For example:

```python
chat_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a Walmart customer support assistant.

Use the provided context to answer the customer's question.
If the context does not contain enough information,
ask for additional information.

Context:
{context}"""
    ),
    (
        "human",
        "{customer_question}"
    )
])
```

The values can then be supplied dynamically:

```python
messages = chat_template.format_messages(
    context="""
    Returns are subject to applicable eligibility conditions.
    Some product categories have different requirements.
    If eligibility cannot be determined, additional information
    is required.
    """,
    customer_question="Can I return an opened laptop after 60 days?"
)
```

The above template now contains **two variables**:

```text
{context}
{customer_question}
```

In [22]:
# Import ChatPromptTemplate from LangChain.
# It allows us to construct prompts using different
# conversation roles such as system and human.
from langchain_core.prompts import ChatPromptTemplate

In [23]:
# Create a structured chat prompt.
chat_prompt = ChatPromptTemplate.from_messages([

    # SYSTEM message:
    # Defines the AI assistant's role and rules.
    (
        "system",
        """
        You are a Walmart customer support assistant.

        Use only the supplied context.

        Do not invent policies, prices, dates,
        or customer-specific information.

        If the context is insufficient,
        say that additional information is required.
        """
    ),

    # HUMAN message:
    # Contains the information and question provided
    # by the customer at runtime.
    (
        "human",
        """
        Context:
        {context}

        Customer question:
        {question}
        """
    )
])

In [25]:
# Create the structured messages from the ChatPromptTemplate.
# This does NOT call Gemini.
messages = chat_prompt.invoke({
    "context": context,
    "question": customer_question
})

# Display the messages that were created.
print(messages)

messages=[SystemMessage(content='\n        You are a Walmart customer support assistant.\n\n        Use only the supplied context.\n\n        Do not invent policies, prices, dates,\n        or customer-specific information.\n\n        If the context is insufficient,\n        say that additional information is required.\n        ', additional_kwargs={}, response_metadata={}), HumanMessage(content='\n        Context:\n        \nReturns are subject to applicable eligibility conditions.\n\nSome product categories have different requirements.\n\nIf eligibility cannot be determined from the available\ninformation, additional information is required.\n\n\n        Customer question:\n        can i return a used item ?\n        ', additional_kwargs={}, response_metadata={})]


In [26]:
# Send the already-created messages to Gemini.
response = llm.invoke(messages)

# Display Gemini's answer.
print("\nWalmart Assistant:", response.content)

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 43.292824701s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '